# Entscheidungsbaum von Grund auf

**Entropie, Gini-Impurity und Baum-Visualisierung**

In diesem Notebook implementieren wir einen Entscheidungsbaum-Klassifikator komplett ohne externe ML-Bibliotheken.
Wir vergleichen die Split-Kriterien **Gini-Impurity** und **Entropie** und visualisieren den trainierten Baum.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons, make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

from decision_tree import DecisionTree, DecisionTreeNode
from plot_utils import plot_decision_boundary

%matplotlib inline
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

## 1. Daten generieren

Wir erzeugen einen synthetischen 2D-Datensatz mit zwei Klassen, um die Decision Boundary gut visualisieren zu können.

In [ ]:
# Synthetische Daten: zwei Halbmonde (nicht linear trennbar)
X, y = make_moons(n_samples=300, noise=0.25, random_state=42)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

print(f"Trainingsdaten: {X_train.shape[0]} Samples")
print(f"Testdaten:      {X_test.shape[0]} Samples")
print(f"Klassen:        {np.unique(y)}")

In [ ]:
# Daten visualisieren
fig, ax = plt.subplots()
scatter = ax.scatter(X[:, 0], X[:, 1], c=y, cmap='RdYlBu', edgecolors='k')
ax.set_xlabel('Merkmal 1')
ax.set_ylabel('Merkmal 2')
ax.set_title('Synthetischer Datensatz: Two Moons')
plt.colorbar(scatter, label='Klasse')
plt.show()

## 2. Gini-Impurity vs. Entropie

### Theorie

**Gini-Impurity:**
$$G = 1 - \sum_{i=1}^{C} p_i^2$$

Misst die Wahrscheinlichkeit einer Fehlklassifikation, wenn ein zufälliges Element entsprechend der Klassenverteilung gelabelt wird.

**Entropie:**
$$H = -\sum_{i=1}^{C} p_i \log_2(p_i)$$

Misst den Informationsgehalt bzw. die „Unordnung" in den Daten. Höhere Entropie = mehr Unsicherheit.

Beide Kriterien werden minimiert — der Split mit dem höchsten **Information Gain** (Differenz zwischen Eltern- und gewichteter Kind-Impurity) wird gewählt.

In [ ]:
# Impurity-Vergleich: Gini vs. Entropie für binäre Verteilung
p = np.linspace(0.001, 0.999, 200)
gini = 1 - (p**2 + (1-p)**2)
entropy = -(p * np.log2(p) + (1-p) * np.log2(1-p))

fig, ax = plt.subplots()
ax.plot(p, gini, label='Gini-Impurity', linewidth=2)
ax.plot(p, entropy, label='Entropie', linewidth=2)
ax.set_xlabel('p (Anteil Klasse 1)')
ax.set_ylabel('Impurity')
ax.set_title('Gini vs. Entropie für binäre Klassifikation')
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

## 3. Entscheidungsbaum trainieren (Gini)

In [ ]:
# Baum mit Gini-Kriterium
tree_gini = DecisionTree(max_depth=5, min_samples_split=5, criterion='gini')
tree_gini.fit(X_train, y_train)

y_pred_gini = tree_gini.predict(X_test)
acc_gini = accuracy_score(y_test, y_pred_gini)
print(f"Genauigkeit (Gini): {acc_gini:.4f}")
print()
print(classification_report(y_test, y_pred_gini, target_names=['Klasse 0', 'Klasse 1']))

In [ ]:
# Baumstruktur ausgeben
print("Entscheidungsbaum (Gini):")
print("=" * 40)
tree_gini.print_tree()

In [ ]:
# Decision Boundary visualisieren
fig, ax = plt.subplots()
plot_decision_boundary(tree_gini, X_train, y_train, 'Decision Boundary — Gini (max_depth=5)', ax=ax)
plt.show()

## 4. Entscheidungsbaum trainieren (Entropie)

In [ ]:
# Baum mit Entropie-Kriterium
tree_entropy = DecisionTree(max_depth=5, min_samples_split=5, criterion='entropy')
tree_entropy.fit(X_train, y_train)

y_pred_ent = tree_entropy.predict(X_test)
acc_ent = accuracy_score(y_test, y_pred_ent)
print(f"Genauigkeit (Entropie): {acc_ent:.4f}")
print()
print(classification_report(y_test, y_pred_ent, target_names=['Klasse 0', 'Klasse 1']))

In [ ]:
# Baumstruktur ausgeben
print("Entscheidungsbaum (Entropie):")
print("=" * 40)
tree_entropy.print_tree()

In [ ]:
# Decision Boundary visualisieren
fig, ax = plt.subplots()
plot_decision_boundary(tree_entropy, X_train, y_train, 'Decision Boundary — Entropie (max_depth=5)', ax=ax)
plt.show()

## 5. Vergleich: Gini vs. Entropie

Beide Kriterien nebeneinander im direkten Vergleich.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

plot_decision_boundary(tree_gini, X_train, y_train,
                       f'Gini (Genauigkeit: {acc_gini:.3f})', ax=axes[0])
plot_decision_boundary(tree_entropy, X_train, y_train,
                       f'Entropie (Genauigkeit: {acc_ent:.3f})', ax=axes[1])

plt.tight_layout()
plt.show()

## 6. Einfluss der Baumtiefe (max_depth)

Wie wirkt sich die maximale Tiefe auf Overfitting/Underfitting aus?

In [ ]:
depths = [1, 2, 3, 5, 10, 20]
train_scores = []
test_scores = []

for d in depths:
    tree = DecisionTree(max_depth=d, min_samples_split=2, criterion='gini')
    tree.fit(X_train, y_train)
    train_scores.append(accuracy_score(y_train, tree.predict(X_train)))
    test_scores.append(accuracy_score(y_test, tree.predict(X_test)))

fig, ax = plt.subplots()
ax.plot(depths, train_scores, 'o-', label='Training', linewidth=2)
ax.plot(depths, test_scores, 's-', label='Test', linewidth=2)
ax.set_xlabel('max_depth')
ax.set_ylabel('Genauigkeit')
ax.set_title('Einfluss der Baumtiefe auf Overfitting')
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Decision Boundaries für verschiedene Tiefen
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.ravel()

for i, d in enumerate(depths):
    tree = DecisionTree(max_depth=d, min_samples_split=2, criterion='gini')
    tree.fit(X_train, y_train)
    acc = accuracy_score(y_test, tree.predict(X_test))
    plot_decision_boundary(tree, X_train, y_train,
                          f'max_depth={d} (Test-Acc: {acc:.3f})', ax=axes[i])

plt.tight_layout()
plt.show()

## 7. Zusammenfassung

- **Gini-Impurity** und **Entropie** liefern oft sehr ähnliche Ergebnisse; Gini ist rechnerisch etwas effizienter (kein log).
- Der Entscheidungsbaum partitioniert den Merkmalsraum rekursiv in achsenparallele Rechtecke.
- **max_depth** kontrolliert die Komplexität: zu flach → Underfitting, zu tief → Overfitting.
- Die Implementierung kommt komplett ohne `scikit-learn`-Baumklassen aus — nur NumPy und eigene Logik.